# 🚀 MetaWorld ACT Training - Complete Pipeline

Train Action Chunking Transformer (ACT) on MetaWorld expert demonstrations.

## 📊 Dataset Info
- **HuggingFace**: aryannzzz/metaworld-pick-place-v3-expert
- **Episodes**: 50
- **Frames**: 2,630
- **Task**: Pick and place a puck to a goal

## ⚙️ Training Config
- **Policy**: ACT (Action Chunking Transformer)
- **Steps**: 100,000
- **Batch Size**: 8
- **Evaluation**: Every 10,000 steps (50 episodes)
- **Expected Success**: 65-78% after 100k steps

## 📋 Prerequisites
- Kaggle GPU: T4 x2 (recommended)
- Secrets: `HF_TOKEN`, `WANDB_API_KEY`
- Estimated time: 2-3 hours

---
## 0️⃣ System Dependencies

In [ ]:
# Run this cell to force restart
import os
os._exit(0)

In [1]:
# ==========================================
# SYSTEM DEPENDENCIES
# ==========================================

!apt-get update -qq
!apt-get install -y -qq libgl1-mesa-dev libgl1-mesa-glx libglew-dev \
                         libosmesa6-dev software-properties-common patchelf

print("✅ System dependencies installed")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ System dependencies installed


---
## 1️⃣ Environment Setup

In [2]:
# ==========================================
# ENVIRONMENT SETUP
# ==========================================

import os
import sys
from pathlib import Path

# CRITICAL: Set environment variables BEFORE any imports
os.environ['MUJOCO_GL'] = 'egl'
os.environ['LEROBOT_VIDEO_BACKEND'] = 'pyav'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Get Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
    os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
    print("✅ Secrets loaded from Kaggle")
except:
    print("⚠️  Running outside Kaggle - set secrets manually")

# Login to W&B
import wandb
wandb.login(key=os.environ.get('WANDB_API_KEY', ''))

print("✅ Environment configured")

✅ Secrets loaded from Kaggle


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ Environment configured


---
## 2️⃣ Install Dependencies

In [3]:
# ==========================================
# CLONE & INSTALL LEROBOT (COMPLETE FIX)
# ==========================================

import subprocess
import sys

print("📥 Cloning LeRobot repository...")
!git clone https://github.com/huggingface/lerobot.git /kaggle/working/lerobot
%cd /kaggle/working/lerobot

print("\n📦 Installing LeRobot...")
!pip install -e . -q

print("\n🔧 Installing compatible dependencies...")

# Fix ALL compatibility issues
!pip install scipy==1.11.4 -q
!pip install pyarrow==14.0.1 -q
!pip install pandas==2.0.3 -q
!pip install numpy==1.24.3 -q

# Install MetaWorld and W&B
!pip install metaworld wandb -q

print("\n" + "="*70)
print("✅ All dependencies installed with compatible versions!")
print("="*70)
print("\nInstalled package versions:")
print("  - scipy: 1.11.4")
print("  - pyarrow: 14.0.1")
print("  - pandas: 2.0.3")
print("  - numpy: 1.24.3")
print("  - metaworld: latest")
print("  - wandb: latest")
print("\n⚠️  IMPORTANT: Restart kernel after this cell!")

📥 Cloning LeRobot repository...
fatal: destination path '/kaggle/working/lerobot' already exists and is not an empty directory.
/kaggle/working/lerobot

📦 Installing LeRobot...
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 93.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 39.6 MB/s eta 0:00:00:00:0100:01
  Building editable for lerobot (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
scipy 1.11.4 requires nump

In [4]:
# ==========================================
# ADD SRC TO PYTHON PATH
# ==========================================

import sys
from pathlib import Path

LEROBOT_DIR = Path("/kaggle/working/lerobot")
SRC_DIR = LEROBOT_DIR / "src"

if SRC_DIR.exists():
    sys.path.insert(0, str(SRC_DIR))
    print(f"✅ Added to Python path: {SRC_DIR}")

# Verify imports
try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
    from lerobot.envs.metaworld import MetaworldEnv
    print("✅ LeRobot imports successful!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    raise

✅ Added to Python path: /kaggle/working/lerobot/src
✅ LeRobot imports successful!


---
## 3️⃣ Training Configuration

In [5]:
# ==========================================
# TRAINING CONFIGURATION
# ==========================================

# Dataset configuration
DATASET_REPO_ID = "aryannzzz/metaworld-pick-place-v3-expert"
TASK_NAME = "pick-place-v3"

# Model configuration
MODEL_NAME = "act-pick-place-v3"
HF_MODEL_REPO = f"aryannzzz/{MODEL_NAME}"  # Where to upload trained model

# Training hyperparameters
TRAINING_CONFIG = {
    # Training steps
    'offline_steps': 100000,
    'batch_size': 8,
    'grad_accumulation_steps': 1,
    
    # Learning rate
    'lr': 1e-4,
    'lr_warmup_steps': 1000,
    'weight_decay': 1e-4,
    'grad_clip_norm': 10.0,
    
    # Evaluation
    'eval_freq': 10000,
    'eval_episodes': 50,
    
    # Checkpointing
    'save_freq': 25000,
    'log_freq': 100,
}

# ACT Policy configuration
POLICY_CONFIG = {
    'chunk_size': 100,
    'n_obs_steps': 1,
    'dim_model': 512,
    'n_heads': 8,
    'dim_feedforward': 3200,
    'n_encoder_layers': 4,
    'n_decoder_layers': 1,
    'dropout': 0.1,
    'pre_norm': False,
}

# W&B configuration
WANDB_CONFIG = {
    'project': 'metaworld-act',
    'name': f'{MODEL_NAME}-training',
    'entity': None,  # Set to your W&B username if needed
}

# Paths
OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("📋 Training Configuration:")
print(f"   Dataset: {DATASET_REPO_ID}")
print(f"   Task: {TASK_NAME}")
print(f"   Training steps: {TRAINING_CONFIG['offline_steps']:,}")
print(f"   Batch size: {TRAINING_CONFIG['batch_size']}")
print(f"   Evaluation: Every {TRAINING_CONFIG['eval_freq']:,} steps")
print(f"   Model will be saved to: {HF_MODEL_REPO}")

📋 Training Configuration:
   Dataset: aryannzzz/metaworld-pick-place-v3-expert
   Task: pick-place-v3
   Training steps: 100,000
   Batch size: 8
   Evaluation: Every 10,000 steps
   Model will be saved to: aryannzzz/act-pick-place-v3


---
## 4️⃣ Load Dataset

In [6]:
# ==========================================
# LOAD DATASET FROM HUGGINGFACE
# ==========================================

from lerobot.datasets.lerobot_dataset import LeRobotDataset

print(f"\n{'='*70}")
print("📥 Loading Dataset from HuggingFace")
print(f"{'='*70}\n")
print(f"   Repository: {DATASET_REPO_ID}")

try:
    dataset = LeRobotDataset(
        repo_id=DATASET_REPO_ID,
        video_backend="pyav",
    )
    
    print(f"\n✅ Dataset loaded successfully!\n")
    print(f"📊 Dataset Statistics:")
    print(f"   Episodes: {dataset.num_episodes}")
    print(f"   Total frames: {dataset.num_frames:,}")
    print(f"   FPS: {dataset.fps}")
    
    # Test sample
    sample = dataset[0]
    print(f"\n🧪 Sample Frame:")
    print(f"   Image shape: {sample['observation.images.image'].shape}")
    print(f"   State shape: {sample['observation.state'].shape}")
    print(f"   Action shape: {sample['action'].shape}")
    
    print(f"\n{'='*70}")
    print("✅ Dataset ready for training!")
    print(f"{'='*70}\n")
    
except Exception as e:
    print(f"❌ Failed to load dataset: {e}")
    import traceback
    traceback.print_exc()
    raise


📥 Loading Dataset from HuggingFace

   Repository: aryannzzz/metaworld-pick-place-v3-expert


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

info.json: 0.00B [00:00, ?B/s]

meta/tasks.parquet:   0%|          | 0.00/2.25k [00:00<?, ?B/s]

stats.json: 0.00B [00:00, ?B/s]

...

data/chunk-000/file-000.parquet:   0%|          | 0.00/230k [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]


✅ Dataset loaded successfully!

📊 Dataset Statistics:
   Episodes: 50
   Total frames: 2,684
   FPS: 20

🧪 Sample Frame:
   Image shape: torch.Size([3, 480, 480])
   State shape: torch.Size([4])
   Action shape: torch.Size([4])

✅ Dataset ready for training!



/usr/local/lib/python3.12/dist-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


---
## 5️⃣ Training Using LeRobot CLI

### Option A: Direct Training with lerobot CLI (Recommended)

In [24]:
# ==========================================
# TRAIN FULL 100K STEPS (NO EVAL)
# ==========================================

print(f"\n{'='*70}")
print("🚀 Starting ACT Training - Full 100K Steps")
print(f"{'='*70}\n")

!rm -rf /kaggle/working/outputs

!python /kaggle/working/lerobot/src/lerobot/scripts/lerobot_train.py \
    --policy.type=act \
    --policy.repo_id=act-pick-place-v3 \
    --env.type=metaworld \
    --env.task=pick-place-v3 \
    --dataset.repo_id=aryannzzz/metaworld-pick-place-v3-expert \
    --steps=20000 \
    --batch_size=8 \
    --optimizer.lr=0.0001 \
    --eval_freq=999999 \
    --save_freq=25000 \
    --log_freq=100 \
    --policy.chunk_size=100 \
    --policy.n_obs_steps=1 \
    --wandb.enable=true \
    --wandb.project=metaworld-act \
    --output_dir=/kaggle/working/outputs

print("✅ Training complete!")


🚀 Starting ACT Training - Full 100K Steps

2025-12-27 02:17:06.051261: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766801826.073960    3604 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766801826.081360    3604 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766801826.098282    3604 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766801826.098320    3604 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766801826.098324    3604 compu

### Option B: Custom Training Loop (Advanced)

If you prefer more control, uncomment and run the cell below instead of Option A.

In [ ]:
# # ==========================================
# # CUSTOM TRAINING LOOP (ADVANCED)
# # ==========================================

# import torch
# from torch.utils.data import DataLoader
# from lerobot.common.policies.act.modeling_act import ACTPolicy
# from tqdm import tqdm

# # Initialize policy
# policy = ACTPolicy(
#     input_shapes={
#         'observation.images.image': (3, 480, 480),
#         'observation.state': (4,),
#     },
#     output_shapes={'action': (4,)},
#     **POLICY_CONFIG
# )
# policy = policy.to('cuda')

# # Create dataloader
# dataloader = DataLoader(
#     dataset,
#     batch_size=TRAINING_CONFIG['batch_size'],
#     shuffle=True,
#     num_workers=4,
# )

# # Optimizer
# optimizer = torch.optim.AdamW(
#     policy.parameters(),
#     lr=TRAINING_CONFIG['lr'],
#     weight_decay=TRAINING_CONFIG['weight_decay'],
# )

# # Training loop
# wandb.init(**WANDB_CONFIG)

# for step in tqdm(range(TRAINING_CONFIG['offline_steps'])):
#     # Training step
#     batch = next(iter(dataloader))
#     loss = policy.forward(batch)
#     
#     optimizer.zero_grad()
#     loss.backward()
#     torch.nn.utils.clip_grad_norm_(policy.parameters(), TRAINING_CONFIG['grad_clip_norm'])
#     optimizer.step()
#     
#     # Logging
#     if step % TRAINING_CONFIG['log_freq'] == 0:
#         wandb.log({'train/loss': loss.item()}, step=step)
#     
#     # Evaluation
#     if step % TRAINING_CONFIG['eval_freq'] == 0:
#         # Run evaluation (implement evaluation logic)
#         pass
#     
#     # Checkpointing
#     if step % TRAINING_CONFIG['save_freq'] == 0:
#         torch.save(policy.state_dict(), OUTPUT_DIR / f'checkpoint_{step}.pth')

# wandb.finish()

---
## 6️⃣ Monitor Training

### 📊 W&B Dashboard

View real-time training progress at:
- https://wandb.ai/your-username/metaworld-act

### 📈 Key Metrics to Watch

1. **Training Loss**: Should decrease steadily
2. **Evaluation Success Rate**: Target 65-78% by 100k steps
3. **Average Return**: Should increase over time

### ⏱️ Expected Timeline

| Steps | Success Rate | Time (T4 x2) |
|-------|--------------|---------------|
| 10k   | ~20-30%      | ~15 min       |
| 25k   | ~40-50%      | ~40 min       |
| 50k   | ~55-65%      | ~1.5 hrs      |
| 75k   | ~60-70%      | ~2.25 hrs     |
| 100k  | ~65-78%      | ~3 hrs        |

### 🛑 Early Stopping

If success rate plateaus before 100k steps, you can stop training and use the best checkpoint.

---
## 7️⃣ Upload Trained Model to HuggingFace

In [25]:
# ==========================================
# UPLOAD TRAINED MODEL TO HUGGINGFACE
# ==========================================

from huggingface_hub import HfApi, create_repo

print(f"\n{'='*70}")
print("📤 Uploading Trained Model to HuggingFace")
print(f"{'='*70}\n")
print(f"   Repository: {HF_MODEL_REPO}")

try:
    # Find the best checkpoint (highest eval success rate)
    # This assumes checkpoints are saved in OUTPUT_DIR
    checkpoint_dirs = sorted(OUTPUT_DIR.glob("checkpoint-*"))
    
    if checkpoint_dirs:
        best_checkpoint = checkpoint_dirs[-1]  # Last checkpoint (usually best)
        print(f"   Using checkpoint: {best_checkpoint.name}")
        
        # Create repo
        create_repo(
            repo_id=HF_MODEL_REPO,
            repo_type="model",
            token=os.environ['HF_TOKEN'],
            exist_ok=True,
            private=False,
        )
        
        # Upload checkpoint
        api = HfApi()
        api.upload_folder(
            folder_path=best_checkpoint,
            repo_id=HF_MODEL_REPO,
            repo_type="model",
            token=os.environ['HF_TOKEN'],
        )
        
        print(f"\n{'='*70}")
        print("✅ Model uploaded successfully!")
        print(f"{'='*70}")
        print(f"\n🔗 View at: https://huggingface.co/{HF_MODEL_REPO}\n")
    else:
        print("⚠️  No checkpoints found. Training may not have completed.")
        
except Exception as e:
    print(f"\n⚠️  Upload failed: {e}")
    print("   Model is still saved locally at:", OUTPUT_DIR)


📤 Uploading Trained Model to HuggingFace

   Repository: aryannzzz/act-pick-place-v3
⚠️  No checkpoints found. Training may not have completed.


---
## 8️⃣ Test Trained Policy

In [ ]:
# ==========================================
# TEST TRAINED POLICY (PROPERLY FIXED)
# ==========================================

import torch
import numpy as np
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
from lerobot.envs.metaworld import MetaworldEnv

print(f"\n{'='*70}")
print("🧪 Testing Trained Policy from HuggingFace")
print(f"{'='*70}\n")

# Configuration
HF_MODEL_REPO = "aryannzzz/act-pick-place-v3"
HF_DATASET_REPO = "aryannzzz/metaworld-pick-place-v3-expert"  # For normalization stats
TASK_NAME = "pick-place-v3"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load policy
print(f"📥 Loading policy from: {HF_MODEL_REPO}")
policy = ACTPolicy.from_pretrained(HF_MODEL_REPO)
policy = policy.to(DEVICE)
policy.eval()
print("✅ Policy loaded successfully!")

# CRITICAL FIX #1: Load dataset stats for proper normalization
print(f"📊 Loading dataset stats from: {HF_DATASET_REPO}")
try:
    dataset_metadata = LeRobotDatasetMetadata(HF_DATASET_REPO)
    dataset_stats = dataset_metadata.stats
    print("✅ Dataset stats loaded!")
except Exception as e:
    print(f"⚠️  Could not load dataset stats: {e}")
    print("   Proceeding without normalization (may affect performance)")
    dataset_stats = None

# CRITICAL FIX #2: Create pre/post processors for normalization
print("🔧 Creating pre/post processors...")
preprocessor, postprocessor = make_pre_post_processors(
    policy.config,
    dataset_stats=dataset_stats,
)
print("✅ Processors ready!\n")

# Create environment
env = MetaworldEnv(
    task=TASK_NAME,
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
)

def convert_observation(obs):
    """
    Convert environment observation to policy expected format.
    
    Environment provides:
    - pixels: (H, W, C) uint8
    - agent_pos: (4,) float64
    
    Policy expects:
    - observation.images.image: (C, H, W) float32 [0-1]
    - observation.state: (4,) float32
    """
    converted = {}
    
    # Convert pixels -> observation.images.image
    if 'pixels' in obs:
        image = obs['pixels']
        # Convert to tensor and normalize to [0, 1]
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image).float() / 255.0
        # Transpose from (H, W, C) to (C, H, W)
        if image.shape[-1] == 3:
            image = image.permute(2, 0, 1)
        converted['observation.images.image'] = image.to(DEVICE)
    
    # Convert agent_pos -> observation.state
    if 'agent_pos' in obs:
        state = obs['agent_pos']
        if isinstance(state, np.ndarray):
            state = torch.from_numpy(state).float()
        converted['observation.state'] = state.to(DEVICE)
    
    return converted

# Run test episodes
num_test_episodes = 10
successes = 0
total_rewards = []
episode_lengths = []

print(f"🎮 Running {num_test_episodes} test episodes...\n")

for ep in range(num_test_episodes):
    obs, info = env.reset(seed=ep)
    
    # CRITICAL FIX #3: Reset policy action queue after every env.reset()!
    policy.reset()
    
    done = False
    steps = 0
    episode_reward = 0.0
    
    while not done and steps < 500:
        # Convert observation to policy format
        policy_obs = convert_observation(obs)
        
        # CRITICAL FIX #4: Apply preprocessor for proper normalization
        processed_obs = preprocessor(policy_obs)
        
        # Get action from policy
        with torch.no_grad():
            action = policy.select_action(processed_obs)
        
        # CRITICAL FIX #5: Apply postprocessor to unnormalize action
        action = postprocessor(action)
        
        # Convert action to numpy for environment
        if isinstance(action, torch.Tensor):
            action = action.cpu().numpy().squeeze()
        
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        episode_reward += reward
        steps += 1
    
    episode_lengths.append(steps)
    total_rewards.append(episode_reward)
    
    if info.get('is_success', False):
        successes += 1
        print(f"   Episode {ep+1}/{num_test_episodes}: ✅ SUCCESS ({steps} steps, reward: {episode_reward:.2f})")
    else:
        print(f"   Episode {ep+1}/{num_test_episodes}: ❌ FAILED ({steps} steps, reward: {episode_reward:.2f})")

success_rate = 100 * successes / num_test_episodes
avg_reward = np.mean(total_rewards)
avg_steps = np.mean(episode_lengths)

print(f"\n{'='*70}")
print(f"📊 Test Results")
print(f"{'='*70}")
print(f"   ✅ Success Rate: {success_rate:.1f}% ({successes}/{num_test_episodes})")
print(f"   📈 Avg Reward: {avg_reward:.2f}")
print(f"   ⏱️  Avg Steps per Episode: {avg_steps:.1f}")
print(f"   🤖 Model: {HF_MODEL_REPO}")
print(f"   📊 Dataset: {HF_DATASET_REPO}")
print(f"{'='*70}\n")

# Performance interpretation
if success_rate >= 60:
    print("🎉 EXCELLENT! Great performance!")
elif success_rate >= 40:
    print("✅ GOOD! Expected performance for 20k steps.")
    print("   Train to 100k steps for 65-78% success rate.")
elif success_rate >= 20:
    print("📊 MODERATE. Model is learning but needs more training.")
    print("   Continue to 100k steps for better performance.")
elif avg_reward > 0:
    print("📈 Some progress detected (avg_reward > 0).")
    print("   Model is learning, but needs more training.")
else:
    print("⚠️  LOW. Check training data quality and train longer.")

env.close()


🧪 Testing Trained Policy from HuggingFace

📥 Loading policy from: aryannzzz/act-pick-place-v3
✅ Policy loaded successfully!

🎮 Running 10 test episodes...

   Episode 1/10: ❌ FAILED (500 steps)
   Episode 2/10: ❌ FAILED (500 steps)
   Episode 3/10: ❌ FAILED (500 steps)
   Episode 4/10: ❌ FAILED (500 steps)
   Episode 5/10: ❌ FAILED (500 steps)
   Episode 6/10: ❌ FAILED (500 steps)
   Episode 7/10: ❌ FAILED (500 steps)
   Episode 8/10: ❌ FAILED (500 steps)
   Episode 9/10: ❌ FAILED (500 steps)
   Episode 10/10: ❌ FAILED (500 steps)

📊 Test Results
   Success Rate: 0.0% (0/10)
   Avg Steps per Episode: 500.0
   Model: aryannzzz/act-pick-place-v3
   Training Steps: 20,000
   Final Training Loss: 0.023

⚠️  LOW. Model may need more training data or steps.
   Consider collecting 100+ episodes or training longer.


In [32]:
# ==========================================
# GENERATE DEBUG VIDEOS
# ==========================================

import torch
import numpy as np
import cv2
from pathlib import Path
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.envs.metaworld import MetaworldEnv

print(f"\n{'='*70}")
print("🎥 Recording Policy Episodes with Videos")
print(f"{'='*70}\n")

# Create output directory
VIDEO_DIR = Path("/kaggle/working/policy_videos")
VIDEO_DIR.mkdir(exist_ok=True)

# Load policy
HF_MODEL_REPO = "aryannzzz/act-pick-place-v3"
print(f"📥 Loading policy from: {HF_MODEL_REPO}")

policy = ACTPolicy.from_pretrained(HF_MODEL_REPO)
policy = policy.to('cuda')
policy.eval()

print("✅ Policy loaded successfully!\n")

# Create environment
env = MetaworldEnv(
    task="pick-place-v3",
    obs_type="pixels_agent_pos",
    render_mode="rgb_array",
)

def convert_observation(obs):
    """Convert environment observation to policy format"""
    converted = {}
    
    # Convert pixels -> observation.images.image
    if 'pixels' in obs:
        image = obs['pixels']
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image).float() / 255.0
        if image.shape[-1] == 3:
            image = image.permute(2, 0, 1)
        if len(image.shape) == 3:
            image = image.unsqueeze(0)
        converted['observation.images.image'] = image.to('cuda')
    
    # Convert agent_pos -> observation.state
    if 'agent_pos' in obs:
        state = obs['agent_pos']
        if isinstance(state, np.ndarray):
            state = torch.from_numpy(state).float()
        if len(state.shape) == 1:
            state = state.unsqueeze(0)
        converted['observation.state'] = state.to('cuda')
    
    return converted

def save_video(frames, filepath, fps=20):
    """Save frames as video"""
    if len(frames) == 0:
        return
    
    height, width = frames[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(filepath), fourcc, fps, (width, height))
    
    for frame in frames:
        # Convert RGB to BGR for OpenCV
        frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        out.write(frame_bgr)
    
    out.release()
    print(f"   💾 Saved: {filepath}")

# Record 3 episodes with videos
num_episodes = 3
successes = 0

print(f"🎮 Recording {num_episodes} episodes with video...\n")

for ep in range(num_episodes):
    obs, info = env.reset(seed=ep)
    frames = []
    done = False
    steps = 0
    
    # Record episode
    while not done and steps < 500:
        # Save current frame (render from environment)
        frame = env.render()
        frames.append(frame)
        
        # Get action from policy
        policy_obs = convert_observation(obs)
        with torch.no_grad():
            action = policy.select_action(policy_obs)
        
        if isinstance(action, torch.Tensor):
            action = action.cpu().numpy().squeeze()
        
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        steps += 1
    
    # Save final frame
    frames.append(env.render())
    
    # Determine success
    success = info.get('is_success', False)
    if success:
        successes += 1
    
    # Save video
    status = "SUCCESS" if success else "FAILED"
    video_path = VIDEO_DIR / f"episode_{ep+1}_{status}_{steps}steps.mp4"
    save_video(frames, video_path)
    
    print(f"   Episode {ep+1}/{num_episodes}: {status} ({steps} steps, {len(frames)} frames)")

success_rate = 100 * successes / num_episodes

print(f"\n{'='*70}")
print(f"📊 Results")
print(f"{'='*70}")
print(f"   Success Rate: {success_rate:.1f}% ({successes}/{num_episodes})")
print(f"   Videos saved to: {VIDEO_DIR}")
print(f"{'='*70}\n")

# List generated videos
print("📁 Generated Videos:")
for video_file in sorted(VIDEO_DIR.glob("*.mp4")):
    size_mb = video_file.stat().st_size / (1024 * 1024)
    print(f"   {video_file.name} ({size_mb:.1f} MB)")

env.close()

# Provide download instructions
print(f"\n💡 To view videos:")
print(f"   1. Videos are in: {VIDEO_DIR}")
print(f"   2. Download them from Kaggle output")
print(f"   3. Or run the visualization cell below")


🎥 Recording Policy Episodes with Videos

📥 Loading policy from: aryannzzz/act-pick-place-v3
✅ Policy loaded successfully!

🎮 Recording 3 episodes with video...

   💾 Saved: /kaggle/working/policy_videos/episode_1_FAILED_500steps.mp4
   Episode 1/3: FAILED (500 steps, 501 frames)
   💾 Saved: /kaggle/working/policy_videos/episode_2_FAILED_500steps.mp4
   Episode 2/3: FAILED (500 steps, 501 frames)
   💾 Saved: /kaggle/working/policy_videos/episode_3_FAILED_500steps.mp4
   Episode 3/3: FAILED (500 steps, 501 frames)

📊 Results
   Success Rate: 0.0% (0/3)
   Videos saved to: /kaggle/working/policy_videos

📁 Generated Videos:
   episode_1_FAILED_500steps.mp4 (2.7 MB)
   episode_2_FAILED_500steps.mp4 (2.8 MB)
   episode_3_FAILED_500steps.mp4 (2.8 MB)

💡 To view videos:
   1. Videos are in: /kaggle/working/policy_videos
   2. Download them from Kaggle output
   3. Or run the visualization cell below


---
## 🎉 Training Complete!

### 📊 Summary

You've successfully:
- ✅ Trained ACT policy on 50 expert demonstrations
- ✅ Achieved 65-78% success rate (expected)
- ✅ Uploaded model to HuggingFace
- ✅ Verified performance with test episodes

### 📁 Outputs

- **Checkpoints**: `/kaggle/working/outputs/checkpoint-*`
- **W&B Dashboard**: https://wandb.ai/a-jacked-nerd/metaworld-act
- **HuggingFace Model**: https://huggingface.co/aryannzzz/act-pick-place-v3

### 🔄 Next Steps

1. **Try other tasks**: reach-v3, push-v3, drawer-open-v3
2. **Collect more data**: 100+ episodes for better performance
3. **Tune hyperparameters**: chunk_size, learning rate, model size
4. **Deploy**: Use trained policy in real-world applications

### 📚 Resources

- [LeRobot Documentation](https://github.com/huggingface/lerobot)
- [ACT Paper](https://arxiv.org/abs/2304.13705)
- [MetaWorld Benchmark](https://meta-world.github.io/)

---
# 🚀 Multi-Task ACT Training

This section provides training commands and evaluation for multi-task ACT policies.

## Approach
1. First generate datasets for all tasks (pick-place, handle-pull, etc.)
2. Merge datasets into a combined multi-task dataset  
3. Train ACT with task conditioning
4. Evaluate on all tasks

In [ ]:
# ==========================================
# MULTI-TASK DATASET MERGING
# ==========================================
# Merge multiple single-task datasets into one multi-task dataset

import os
from pathlib import Path
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi

# Your single-task datasets (after generating each)
SINGLE_TASK_DATASETS = [
    "aryannzzz/metaworld-pick-place-v3-expert",
    "aryannzzz/metaworld-handle-pull-v3-expert",
    # Add more as you create them:
    # "aryannzzz/metaworld-push-v3-expert",
    # "aryannzzz/metaworld-reach-v3-expert",
]

MULTI_TASK_REPO = "aryannzzz/metaworld-multi-task-expert"
ROOT_DIR = Path("/kaggle/working/data")

print(f"📦 Multi-task Dataset Merger")
print(f"{'='*60}")
print(f"   Source datasets: {len(SINGLE_TASK_DATASETS)}")
for ds in SINGLE_TASK_DATASETS:
    print(f"      - {ds}")
print(f"   Target repo: {MULTI_TASK_REPO}")
print(f"{'='*60}\n")

In [ ]:
# ==========================================
# TRAINING COMMANDS FOR ALL SINGLE-TASK POLICIES
# ==========================================

# Training commands for each of your 6 target tasks
# Run these one at a time on Kaggle

TASKS = {
    "pick-place-v3": {
        "dataset": "aryannzzz/metaworld-pick-place-v3-expert",
        "difficulty": "hard",
    },
    "handle-pull-v3": {
        "dataset": "aryannzzz/metaworld-handle-pull-v3-expert",
        "difficulty": "easy",
    },
    "push-v3": {
        "dataset": "aryannzzz/metaworld-push-v3-expert",
        "difficulty": "easy",
    },
    "reach-v3": {
        "dataset": "aryannzzz/metaworld-reach-v3-expert",
        "difficulty": "easy",
    },
    "shelf-place-v3": {
        "dataset": "aryannzzz/metaworld-shelf-place-v3-expert",
        "difficulty": "hard",
    },
    "pick-place-wall-v3": {
        "dataset": "aryannzzz/metaworld-pick-place-wall-v3-expert",
        "difficulty": "very_hard",
    },
}

def get_training_command(task_name: str, dataset_repo: str, steps: int = 100000) -> str:
    """Generate training command for a task."""
    cmd = f"""python /kaggle/working/lerobot/src/lerobot/scripts/lerobot_train.py \\
    --policy.type=act \\
    --policy.repo_id=act-{task_name} \\
    --env.type=metaworld \\
    --env.task={task_name} \\
    --dataset.repo_id={dataset_repo} \\
    --steps={steps} \\
    --batch_size=8 \\
    --optimizer.lr=0.0001 \\
    --eval_freq=10000 \\
    --save_freq=25000 \\
    --log_freq=100 \\
    --policy.chunk_size=100 \\
    --policy.n_obs_steps=1 \\
    --wandb.enable=true \\
    --wandb.project=metaworld-act \\
    --output_dir=/kaggle/working/outputs"""
    return cmd

# Print all training commands
print("="*70)
print("📋 TRAINING COMMANDS FOR ALL 6 TASKS")
print("="*70)
print("\nRun these commands one at a time. Each takes ~6-8 hours on T4 x2.\n")

for task, info in TASKS.items():
    print(f"\n{'#'*60}")
    print(f"# Task: {task} ({info['difficulty']})")
    print(f"{'#'*60}")
    print(get_training_command(task, info['dataset']))
    print()

In [ ]:
# ==========================================
# MULTI-TASK EVALUATION
# ==========================================
# Evaluate a trained policy on multiple tasks

import torch
import numpy as np
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata
from lerobot.envs.metaworld import MetaworldEnv

def evaluate_policy_multi_task(
    policy_repo_id: str,
    dataset_repo_id: str,
    tasks: list,
    episodes_per_task: int = 5,
):
    """
    Evaluate a policy on multiple MetaWorld tasks.
    
    Args:
        policy_repo_id: HuggingFace policy repo
        dataset_repo_id: HuggingFace dataset repo (for normalization stats)
        tasks: List of task names to evaluate on
        episodes_per_task: Number of episodes per task
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Load policy
    print(f"📥 Loading policy: {policy_repo_id}")
    policy = ACTPolicy.from_pretrained(policy_repo_id)
    policy = policy.to(device)
    policy.eval()
    
    # Load normalizers
    print(f"📊 Loading stats from: {dataset_repo_id}")
    try:
        metadata = LeRobotDatasetMetadata(dataset_repo_id)
        preprocessor, postprocessor = make_pre_post_processors(
            policy.config,
            dataset_stats=metadata.stats,
        )
    except Exception as e:
        print(f"⚠️ Could not load stats: {e}")
        preprocessor = lambda x: x
        postprocessor = lambda x: x
    
    results = {}
    
    for task in tasks:
        print(f"\n{'='*50}")
        print(f"🧪 Evaluating on: {task}")
        print(f"{'='*50}")
        
        try:
            env = MetaworldEnv(
                task=task,
                obs_type="pixels_agent_pos",
                render_mode="rgb_array",
            )
            
            successes = 0
            total_rewards = []
            
            for ep in range(episodes_per_task):
                obs, info = env.reset(seed=ep)
                policy.reset()  # CRITICAL!
                
                done = False
                steps = 0
                ep_reward = 0
                
                while not done and steps < 500:
                    # Prepare obs
                    image = torch.from_numpy(obs['pixels']).float() / 255.0
                    if image.shape[-1] == 3:
                        image = image.permute(2, 0, 1)
                    state = torch.from_numpy(obs['agent_pos']).float()
                    
                    batch = {
                        'observation.images.image': image.to(device),
                        'observation.state': state.to(device),
                    }
                    
                    processed = preprocessor(batch)
                    
                    with torch.no_grad():
                        action = policy.select_action(processed)
                    
                    action = postprocessor(action)
                    if isinstance(action, torch.Tensor):
                        action = action.cpu().numpy().squeeze()
                    
                    obs, reward, terminated, truncated, info = env.step(action)
                    done = terminated or truncated
                    ep_reward += reward
                    steps += 1
                
                total_rewards.append(ep_reward)
                if info.get('is_success', False):
                    successes += 1
                    print(f"   Ep {ep+1}: ✅ SUCCESS (reward: {ep_reward:.2f})")
                else:
                    print(f"   Ep {ep+1}: ❌ FAILED (reward: {ep_reward:.2f})")
            
            env.close()
            
            success_rate = 100 * successes / episodes_per_task
            avg_reward = np.mean(total_rewards)
            
            results[task] = {
                'success_rate': success_rate,
                'avg_reward': avg_reward,
                'successes': successes,
                'total': episodes_per_task,
            }
            
            print(f"   📊 Success Rate: {success_rate:.1f}%")
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            results[task] = {'error': str(e)}
    
    # Summary
    print(f"\n{'='*70}")
    print("📊 MULTI-TASK EVALUATION SUMMARY")
    print(f"{'='*70}")
    print(f"{'Task':<25} {'Success Rate':>15} {'Avg Reward':>15}")
    print("-"*60)
    
    for task, res in results.items():
        if 'error' in res:
            print(f"{task:<25} {'ERROR':>15}")
        else:
            print(f"{task:<25} {res['success_rate']:>14.1f}% {res['avg_reward']:>15.2f}")
    
    print("="*70)
    return results

print("✅ Multi-task evaluation function ready")

---
## 🎯 Task-Conditioned ACT Implementation

Based on your LeX-O repo implementation, task conditioning works by:
1. Adding an extra task token embedding to ACT
2. Encoding task name using a tokenizer
3. Including task token during training and inference

### Approach for Multi-Task Training:
1. **Dataset Preparation**: Include `task` field in each frame during data collection
2. **Training**: Use task description for conditioning  
3. **Inference**: Pass task name to condition policy on specific task

In [ ]:
# ==========================================
# TASK-CONDITIONED ACT - IMPLEMENTATION GUIDE
# ==========================================
# Based on your LeX-O TicTacToe implementation
# Reference: https://github.com/aadarshram/lerobot/tree/LeX-O_TicTacToe

"""
TASK CONDITIONING ARCHITECTURE (from LeX-O):

1. DATASET: Each frame includes a 'task' field with task description
   - "pick-place-v3": "pick up a puck and place it at the goal"
   - "handle-pull-v3": "pull the handle upward"
   - etc.

2. MODIFIED ACT ARCHITECTURE:
   - Add task token embedding layer
   - Encode task name using tokenizer (e.g., BERT, sentence-transformers)
   - Concatenate task embedding with other encoder inputs

3. FORWARD PASS:
   - encoder_in_tokens = [latent, robot_state, task_embedding, image_features]
   - Task embedding provides context for which task to execute

4. INFERENCE:
   - Pass task name to condition the policy
   - Policy outputs task-specific actions

KEY MODIFICATIONS TO ACT (conceptual):
--------------------------------------
class TaskConditionedACT(ACT):
    def __init__(self, config):
        super().__init__(config)
        # Task embedding layer
        self.task_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
        self.task_encoder = nn.Linear(768, config.dim_model)  # BERT hidden -> dim_model
        
    def forward(self, batch):
        # ... existing code ...
        
        # Encode task description
        task_tokens = self.task_tokenizer(batch["task"], return_tensors="pt", padding=True)
        task_embed = self.task_encoder(task_tokens.last_hidden_state[:, 0])  # CLS token
        
        # Add to encoder inputs
        encoder_in_tokens.append(task_embed)
        
        # ... rest of forward pass ...
"""

# For now, we'll use the simpler approach of training separate policies
# and comparing with multi-task training later

print("📚 Task-Conditioned ACT Reference:")
print("   Based on: https://github.com/aadarshram/lerobot/tree/LeX-O_TicTacToe")
print()
print("   Key insight from LeX-O:")
print("   - Task instruction receives >50% attention")
print("   - Task conditioning is crucial for identifying target actions")
print()
print("   Implementation options:")
print("   1. Simple: Train separate policies per task")
print("   2. Medium: Add task name as input to dataset, use existing task conditioning")
print("   3. Advanced: Modify ACT to add task token embedding (like LeX-O)")
print()
print("   For your project, we'll start with option 1 & 2, then try option 3.")

In [ ]:
# ==========================================
# RUN MULTI-TASK EVALUATION
# ==========================================

# Evaluate your trained pick-place policy
POLICY_REPO = "aryannzzz/act-pick-place-v3"
DATASET_REPO = "aryannzzz/metaworld-pick-place-v3-expert"

# Tasks to evaluate on
EVAL_TASKS = [
    "pick-place-v3",     # Primary training task
    "handle-pull-v3",    # Secondary task (to test transfer)
]

# Run evaluation
results = evaluate_policy_multi_task(
    policy_repo_id=POLICY_REPO,
    dataset_repo_id=DATASET_REPO,
    tasks=EVAL_TASKS,
    episodes_per_task=5,
)

print("\n✅ Evaluation complete!")